In [ ]:
import glob
import os
import sys

import einops
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from kornia.geometry.transform import resize
from matplotlib import pyplot as plt
from PIL import Image
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

sys.path.append("your_path")
from src.tasks.pretrain.backbone import (
    get_retfound,
)
from src.tasks.pretrain.encoder import SSLEncoder
from src.tasks.pretrain.utils import (
    compute_pca_features,
    min_max_scale_2d,
)

# Set random seeds for reproducibility
torch.manual_seed(123)
np.random.seed(123)

# Set CUDA device
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Change this to the GPU you want to use

# Model Loader

In [ ]:
def get_freqmae(weights_path):
    encoder = SSLEncoder(
        model_name="timm/vit_base_patch14_reg4_dinov2.lvd142m",
        algorithm="FmiMAE",
        pretrained=True,
        ckpt_path=weights_path,
        mask_ratio=0.0,
        img_size=224,
        patch_size=14,
        reg_tokens=4,
        in_chans=3,
        embed_dim=768,
        depth=12,
        num_heads=12,
        mlp_ratio=4.0,
        drop_rate=0.0,
        pos_drop_rate=0.0,
        patch_drop_rate=0.0,
        proj_drop_rate=0.0,
        attn_drop_rate=0.0,
        drop_path_rate=0.0,
        standardize=True,
        dynamic_img_size=True,
        norm_layer=nn.LayerNorm,
    )
    return encoder.backbone

# Data Loader

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, img_paths, transform=None, img_size=224):
        """
        Args:
            img_dir (str): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.img_paths = img_paths
        self.transform = transform or transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Resize((img_size, img_size), antialias=True),
            ]
        )

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image

# Main Functions

In [ ]:
def patch_pca_feats(patch_feats, patch_size, img, mode="nearest"):
    if patch_feats.ndim == 3:
        B, N, D = patch_feats.shape
        patch_feats = einops.rearrange(patch_feats, "b (h w) c -> (b h w) c", h=int(np.sqrt(N)))

    elif patch_feats.ndim == 4:
        B, D, H, W = patch_feats.shape
        N = H * W
        patch_feats = einops.rearrange(patch_feats, "b c h w -> (b h w) c")

    fg_mask = img.mean(dim=1)
    th = 0.0868832016680601 * fg_mask.amax(dim=(1, 2), keepdim=True)
    fg_mask = (fg_mask > th).float()
    fg_mask = F.interpolate(
        fg_mask.unsqueeze(1), size=(int(np.sqrt(N)), int(np.sqrt(N))), mode="nearest"
    )
    fg_mask = fg_mask.squeeze(1)
    # fg_mask = einops.rearrange(fg_mask, "b c h w -> (b h w) c", c=1).bool().repeat(1, D)

    # PCA 특징 계산
    pca_feats = compute_pca_features(patch_feats, fg_mask=fg_mask)  # ((b h w) c)
    h = w = int(np.sqrt(N))
    pca_feats = einops.rearrange(pca_feats, "(b h w) c -> b h w c", h=h, w=w).permute(0, 3, 1, 2)

    # 업샘플링
    target_size = (h * patch_size, w * patch_size)
    pca_feats = torch.nn.functional.interpolate(
        pca_feats, size=target_size, mode=mode
    )  # (B, 3, H, W)

    return pca_feats.clamp(0, 1)

# Model Load

In [ ]:
# 모델 로드
freqmae = get_freqmae(
    "your_path/save/fundus_ssl/logs/ssl-freqmae-vit_base_224_14-42/runs/m_0.2_hf_0.1_t_0.1_flair/checkpoints/encoder_decoder-epoch_0000099.ckpt"
)
retfound = get_retfound()

## PCA feats

In [ ]:
# 이미지 디렉토리 경로 설정
img_paths_dr = [
    "your_path/src/tasks/pretrain/visualization/imgs/IDRiD_ADR.png",
    "your_path/src/tasks/pretrain/visualization/imgs/IDRiD_ADR2.png",
    "your_path/src/tasks/pretrain/visualization/imgs/IDRiD_EDR.png",
    "your_path/src/tasks/pretrain/visualization/imgs/IDRiD_EDR3.png",
]
# 데이터셋 및 데이터로더 생성
dataset_dr = ImageDataset(img_paths_dr, img_size=224)
dataloader_dr = DataLoader(dataset_dr, batch_size=8, shuffle=False)

# 모든 이미지를 하나의 텐서로 로드
dr_images = []
for batch in dataloader_dr:
    dr_images.append(batch)

# 모든 배치를 하나의 텐서로 결합 (B, C, H, W)
data_dr = torch.cat(dr_images, dim=0)
print(f"Loaded images tensor shape: {data_dr.shape}")

# 시각화를 위해 첫 번째 이미지 확인 (선택 사항)
if len(data_dr) > 0:
    print(f"Number of images loaded: {len(data_dr)}")
    print(f"Image dimensions: {data_dr[0].shape}")


layer_name = "late"

# Get features for current batch
feats_freqmae = freqmae.get_intermediate_features(data_dr)[layer_name]
feats_retfound = retfound.get_intermediate_features(data_dr)[layer_name]

# Apply PCA to features
pca_feats_freqmae = patch_pca_feats(feats_freqmae[:, 5:], 16, data_dr)
pca_feats_retfound = patch_pca_feats(feats_retfound[:, 1:], 16, data_dr)

feats = {
    "RetFound": pca_feats_retfound,
    "FunMMA": pca_feats_freqmae,
}

# ─── 출력 디렉토리 생성 ─────────────────────────────────────────────
output_dir = (
    "your_path/src/tasks/pretrain/visualization/feats/pca_feats/dr"
)
os.makedirs(output_dir, exist_ok=True)


def save_rgb(arr, path):
    """
    arr: torch.Tensor or numpy array of shape (3, H, W) or (H, W, 3)
    path: where to save the PNG
    norm_along_channel: if True, perform min-max normalization per channel;
                        otherwise normalize across all channels together.
    """
    # 1) torch.Tensor → numpy 변환
    if isinstance(arr, torch.Tensor):
        arr = arr.detach().cpu().numpy()
    # 2) (3, H, W) → (H, W, 3)
    if arr.ndim == 3 and arr.shape[0] == 3:
        arr = arr.transpose(1, 2, 0)
    # 3) 정규화 to [0,1]

    mn, mx = arr.min(), arr.max()
    if mx > mn:
        norm = (arr - mn) / (mx - mn)
    else:
        norm = np.zeros_like(arr, dtype=np.float32)
    # 4) uint8로 변환 및 저장
    img_uint8 = (norm * 255).astype(np.uint8)
    Image.fromarray(img_uint8).save(path)


# ─── 1) PCA feature에 대해 채널별 min-max 정규화 후 저장 ────────────────────────
for name, array in feats.items():
    # array: (B, C, H, W)
    # 채널별로 배치 전체 및 공간 차원에서 min/max 계산
    mins = array.min(dim=(0, 2, 3), keepdim=True)[0]  # shape (1, C, 1, 1)
    maxs = array.max(dim=(0, 2, 3), keepdim=True)[0]  # shape (1, C, 1, 1)
    denom = (maxs - mins).clamp(min=1e-6)  # 0 방지
    norm_array = (array - mins) / denom  # shape (B, C, H, W)

    B, C, H, W = norm_array.shape
    for i in range(B):
        save_rgb(
            norm_array[i].cpu().numpy(),
            os.path.join(output_dir, f"{name}_batch{i}.png"),
            norm_along_channel=False,
        )

# ─── 2) 원본 입력 RGB 이미지는 기존대로 저장 ─────────────────────────────────
B, C, H, W = data_dr.shape
for i in range(B):
    save_rgb(data_dr[i], os.path.join(output_dir, f"input_batch{i}.png"), norm_along_channel=False)

print(f"All images saved under {output_dir}")

In [ ]:
# 이미지 디렉토리 경로 설정
img_paths_amd = [
    "your_path/src/tasks/pretrain/visualization/imgs/RFMiD_DAMD.png",
    "your_path/src/tasks/pretrain/visualization/imgs/RFMiD_DAMD2.png",
    "your_path/src/tasks/pretrain/visualization/imgs/RFMiD_WAMD.png",
    "your_path/src/tasks/pretrain/visualization/imgs/RFMiD_WAMD2.png",
]
# 데이터셋 및 데이터로더 생성
dataset_amd = ImageDataset(img_paths_amd, img_size=224)
dataloader_amd = DataLoader(dataset_amd, batch_size=8, shuffle=False)

# 모든 이미지를 하나의 텐서로 로드
amd_images = []
for batch in dataloader_amd:
    amd_images.append(batch)

# 모든 배치를 하나의 텐서로 결합 (B, C, H, W)
data_amd = torch.cat(amd_images, dim=0)
print(f"Loaded images tensor shape: {data_amd.shape}")

# 시각화를 위해 첫 번째 이미지 확인 (선택 사항)
if len(data_amd) > 0:
    print(f"Number of images loaded: {len(data_amd)}")
    print(f"Image dimensions: {data_amd[0].shape}")


layer_name = "late"
output_dir = (
    "your_path/src/tasks/pretrain/visualization/feats/pca_feats/amd"
)
os.makedirs(output_dir, exist_ok=True)

# Get features for current batch
feats_freqmae = freqmae.get_intermediate_features(data_amd)[layer_name]
feats_retfound = retfound.get_intermediate_features(data_amd)[layer_name]

# Apply PCA to features
pca_feats_freqmae = patch_pca_feats(feats_freqmae[:, 5:], 16, data_amd)
pca_feats_retfound = patch_pca_feats(feats_retfound[:, 1:], 16, data_amd)

feats = {
    "RetFound": pca_feats_retfound,
    "FunMMA": pca_feats_freqmae,
}


def save_rgb(arr, path):
    """
    arr: torch.Tensor or numpy array of shape (3, H, W) or (H, W, 3)
    path: where to save the PNG
    norm_along_channel: if True, perform min-max normalization per channel;
                        otherwise normalize across all channels together.
    """
    # 1) torch.Tensor → numpy 변환
    if isinstance(arr, torch.Tensor):
        arr = arr.detach().cpu().numpy()
    # 2) (3, H, W) → (H, W, 3)
    if arr.ndim == 3 and arr.shape[0] == 3:
        arr = arr.transpose(1, 2, 0)
    # 3) 정규화 to [0,1]

    mn, mx = arr.min(), arr.max()
    if mx > mn:
        norm = (arr - mn) / (mx - mn)
    else:
        norm = np.zeros_like(arr, dtype=np.float32)
    # 4) uint8로 변환 및 저장
    img_uint8 = (norm * 255).astype(np.uint8)
    Image.fromarray(img_uint8).save(path)


# ─── 1) PCA feature에 대해 채널별 min-max 정규화 후 저장 ────────────────────────
for name, array in feats.items():
    # array: (B, C, H, W)
    # 채널별로 배치 전체 및 공간 차원에서 min/max 계산
    mins = array.min(dim=(0, 2, 3), keepdim=True)[0]  # shape (1, C, 1, 1)
    maxs = array.max(dim=(0, 2, 3), keepdim=True)[0]  # shape (1, C, 1, 1)
    denom = (maxs - mins).clamp(min=1e-6)  # 0 방지
    norm_array = (array - mins) / denom  # shape (B, C, H, W)

    B, C, H, W = norm_array.shape
    for i in range(B):
        save_rgb(
            norm_array[i].cpu().numpy(),
            os.path.join(output_dir, f"{name}_batch{i}.png"),
            norm_along_channel=False,
        )

# ─── 2) 원본 입력 RGB 이미지는 기존대로 저장 ─────────────────────────────────
B, C, H, W = data_dr.shape
for i in range(B):
    save_rgb(data_dr[i], os.path.join(output_dir, f"input_batch{i}.png"), norm_along_channel=False)

print(f"All images saved under {output_dir}")